In [1]:
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

# =========================================================
# 0. 원본 로드
# =========================================================
portfolio = pd.read_json("portfolio.json", lines=True)
profile = pd.read_json("profile.json", lines=True)
transcript = pd.read_json("transcript.json", lines=True)

print("원본 shape:", portfolio.shape, profile.shape, transcript.shape)


# =========================================================
# 1. portfolio_전처리
#    - 오퍼 자체의 "스펙"을 정리하는 테이블
# =========================================================
portfolio_전처리 = portfolio.copy()

# (1) id -> offer_id : 나중에 transcript와 합칠 때 이름을 통일해야 merge가 됨
portfolio_전처리 = portfolio_전처리.rename(columns={"id": "offer_id"})

# (2) channels 리스트 -> 원-핫 컬럼 4개
#     ["email","mobile"] 같은 리스트 형태는 필터링/통계가 안 되므로
#     is_web/is_email/is_mobile/is_social 0/1 컬럼으로 펼침
for ch in ["web", "email", "mobile", "social"]:
    portfolio_전처리[f"is_{ch}"] = portfolio_전처리["channels"].apply(lambda lst: int(ch in lst))
portfolio_전처리 = portfolio_전처리.drop(columns=["channels"])

# (3) duration(일) -> duration_hours(시간)
#     transcript의 time이 "시작 후 몇 시간"이라는 시간 단위라서,
#     오퍼 유효기간도 같은 단위(시간)로 맞춰야 나중에 시간창 매칭이 가능함
portfolio_전처리["duration_hours"] = portfolio_전처리["duration"] * 24

# (4) reward_ratio = reward / difficulty
#     "이 오퍼가 완료됐을 때, 매출 대비 몇 %를 회사가 되돌려주는가" 비용율 지표
#     informational은 difficulty=0이라 나누면 0/0(정의 불가) -> NaN으로 명시적 결측 처리
portfolio_전처리["reward_ratio"] = np.where(
    portfolio_전처리["difficulty"] > 0,
    portfolio_전처리["reward"] / portfolio_전처리["difficulty"],
    np.nan,
)

print("\n[portfolio_전처리]")
print(portfolio_전처리.head())


# =========================================================
# 2. profile_전처리
#    - 고객 인구통계 테이블
# =========================================================
profile_전처리 = profile.copy()

# (1) id -> person : transcript와 이름 통일
profile_전처리 = profile_전처리.rename(columns={"id": "person"})

# (2) age=118(결측 placeholder) -> missing_profile 플래그로 통합
#     age=118인 사람은 gender/income도 항상 같이 비어있는 걸 확인했으므로
#     3개 컬럼 결측을 따로 다루지 않고 "프로필 미등록"이라는 하나의 이분값으로 정리
profile_전처리["missing_profile"] = (profile_전처리["age"] == 118).astype(int)
profile_전처리.loc[profile_전처리["missing_profile"] == 1, "age"] = np.nan

# (3) income -> 사분위수 기준 소득구간
#     연속형 income을 4개 그룹으로 나눠 그룹 간 비교(예: 소득구간별 완료율)를 할 수 있게 함
#     결측(NaN)은 구간 계산에서 자동으로 제외되어 NaN으로 남음
profile_전처리["income_quartile"] = pd.qcut(
    profile_전처리["income"], q=4, labels=["Q1_저", "Q2_중하", "Q3_중상", "Q4_고"]
)

# (4) became_member_on(날짜) -> 가입경과일(숫자)
#     날짜 형태는 통계 연산이 안 되므로, 데이터 내 최신 시점 기준 "가입 후 며칠째"로 변환
profile_전처리["became_member_on"] = pd.to_datetime(
    profile_전처리["became_member_on"], format="%Y%m%d"
)
reference_date = profile_전처리["became_member_on"].max()
profile_전처리["membership_days"] = (
    reference_date - profile_전처리["became_member_on"]
).dt.days

# (5) gender='O'는 별도 결측 처리 없이 그대로 유지(하나의 범주로만 취급, 표본 작아 세부분석 제외)

print("\n[profile_전처리]")
print(profile_전처리.head())
print("\nmissing_profile 분포:\n", profile_전처리["missing_profile"].value_counts())


# =========================================================
# 3. transcript_전처리
#    - event 4종류가 뒤섞인 로그를, 종류별로 분리 + value 딕셔너리를 컬럼으로 펼침
# =========================================================
def unpack_value(row):
    """value 딕셔너리 안의 키가 이벤트마다 달라서(offer id/offer_id/amount/reward)
    하나로 통일해서 꺼내는 함수"""
    v = row["value"]
    offer_id = v.get("offer id", v.get("offer_id"))
    amount = v.get("amount")
    reward = v.get("reward")
    return pd.Series({"offer_id": offer_id, "amount": amount, "reward_received": reward})

transcript_전처리 = transcript.copy()
transcript_전처리 = pd.concat(
    [transcript_전처리, transcript_전처리.apply(unpack_value, axis=1)], axis=1
).drop(columns=["value"])

# event 종류별로 4개 테이블로 분리
received_df = transcript_전처리[transcript_전처리["event"] == "offer received"][
    ["person", "offer_id", "time"]
].rename(columns={"time": "received_time"})

viewed_df = transcript_전처리[transcript_전처리["event"] == "offer viewed"][
    ["person", "offer_id", "time"]
].rename(columns={"time": "viewed_time"})

completed_df = transcript_전처리[transcript_전처리["event"] == "offer completed"][
    ["person", "offer_id", "time", "reward_received"]
].rename(columns={"time": "completed_time"})

# 완료 이벤트 중 완전 중복(person, offer_id, time 동일) 제거
before = len(completed_df)
completed_df = completed_df.drop_duplicates(subset=["person", "offer_id", "completed_time"])
print(f"\noffer completed 중복 제거: {before} -> {len(completed_df)} "
      f"({before - len(completed_df)}건 제거)")

transaction_df = transcript_전처리[transcript_전처리["event"] == "transaction"][
    ["person", "time", "amount"]
].rename(columns={"time": "transaction_time"})

print("\n[transcript_전처리 - event별 행 수]")
print("received:", len(received_df), "/ viewed:", len(viewed_df),
      "/ completed(정제후):", len(completed_df), "/ transaction:", len(transaction_df))


# =========================================================
# 4. 오퍼 인스턴스 테이블 만들기 (3개 전처리 테이블 병합)
#    - "받은 사건 1건 = 인스턴스 1행"이 기본 단위
#    - 같은 (person, offer_id)를 여러 번 받은 경우, 다음 수신 시점 전까지만
#      그 인스턴스의 유효 구간으로 잡아 재수신 매칭 오류를 방지
# =========================================================

# (1) 인스턴스 기본 뼈대: received_df 자체가 곧 인스턴스 단위 (76,277건)
instance = received_df.copy()
instance["instance_id"] = np.arange(len(instance))

# (2) portfolio 스펙 붙이기 (offer_id 기준)
instance = instance.merge(portfolio_전처리, on="offer_id", how="left")

# (3) profile 인구통계 붙이기 (person 기준)
instance = instance.merge(profile_전처리, on="person", how="left")

# (4) 이 인스턴스의 유효 시간창 계산
#     window_end = min(received_time + duration_hours, 같은 (person,offer_id)의 다음 수신 시각)
#     is_capped: window_end가 "재수신 때문에 잘린 경계"인지, "자연 만료"인지 구분
#       - capping된 경계는 다음 인스턴스의 received_time과 정확히 같은 값이라,
#         등호(<=)를 허용하면 그 경계에 걸린 이벤트가 양쪽 인스턴스에 동시 매칭(이중집계)되어버림
#       - 자연 만료 경계는 다음 인스턴스가 아예 없거나 멀리 있어, 등호를 허용해도 안전함
instance = instance.sort_values(["person", "offer_id", "received_time"]).reset_index(drop=True)
instance["next_received_time"] = (
    instance.groupby(["person", "offer_id"])["received_time"].shift(-1)
)
instance["natural_end"] = instance["received_time"] + instance["duration_hours"]
instance["window_end"] = instance[["natural_end", "next_received_time"]].min(axis=1)
instance["is_capped"] = instance["window_end"] == instance["next_received_time"]
# (참고) NaN == NaN은 항상 False이므로, next_received_time이 없는(재수신 없는 마지막) 인스턴스는
# 이 조건에서 자동으로 False(=자연만료)로 분류됨. 별도의 자연만료 비교 조건은 필요 없으며,
# 오히려 "자연만료 시점과 다음 수신 시점이 우연히 같은 값"인 경계 케이스를 놓치는 원인이 되므로 제거함.

# (5) viewed 매칭
#     capping된 인스턴스는 received_time <= viewed_time < window_end (이중매칭 방지)
#     자연만료 인스턴스는 received_time <= viewed_time <= window_end (경계 동타 살림)
merged_view = instance.merge(viewed_df, on=["person", "offer_id"], how="left")
in_range_v = merged_view["viewed_time"] >= merged_view["received_time"]
end_ok_v = np.where(
    merged_view["is_capped"],
    merged_view["viewed_time"] < merged_view["window_end"],
    merged_view["viewed_time"] <= merged_view["window_end"],
)
mask_v = in_range_v & end_ok_v
first_view = (
    merged_view[mask_v]
    .groupby("instance_id")["viewed_time"]
    .min()
    .rename("viewed_time")
)
instance = instance.merge(first_view, on="instance_id", how="left")
instance["is_viewed"] = instance["viewed_time"].notna().astype(int)

# (6) completed 매칭 (viewed와 동일한 조건부 <= 로직)
merged_comp = instance.merge(completed_df, on=["person", "offer_id"], how="left")
in_range_c = merged_comp["completed_time"] >= merged_comp["received_time"]
end_ok_c = np.where(
    merged_comp["is_capped"],
    merged_comp["completed_time"] < merged_comp["window_end"],
    merged_comp["completed_time"] <= merged_comp["window_end"],
)
mask_c = in_range_c & end_ok_c
first_comp = (
    merged_comp[mask_c]
    .groupby("instance_id")
    .agg(completed_time=("completed_time", "min"), reward_received=("reward_received", "first"))
)
instance = instance.merge(first_comp, on="instance_id", how="left")
instance["is_completed"] = instance["completed_time"].notna().astype(int)

# (7) transaction 매칭: completed_time과 "같은 시각(person, time)"에 찍힌 transaction 금액 귀속
tx_matched = instance[["instance_id", "person", "completed_time"]].dropna(subset=["completed_time"])
tx_matched = tx_matched.merge(
    transaction_df, left_on=["person", "completed_time"], right_on=["person", "transaction_time"], how="left"
)
# 한 거래가 여러 오퍼 완료를 동시에 만족시킨 경우 금액이 중복 집계될 수 있어 플래그로 표시
tx_matched["dup_transaction_flag"] = tx_matched.duplicated(subset=["person", "transaction_time"], keep=False).astype(int)
tx_matched = tx_matched[["instance_id", "amount", "dup_transaction_flag"]].rename(
    columns={"amount": "matched_amount"}
)
instance = instance.merge(tx_matched, on="instance_id", how="left")

instance = instance.drop(columns=["next_received_time"])

print("\n=== 최종 오퍼 인스턴스 테이블 ===")
print("shape:", instance.shape)
print("is_viewed 합:", instance["is_viewed"].sum(), "(원본 offer viewed 행수:", len(viewed_df), ")")
print("is_completed 합:", instance["is_completed"].sum(), "(정제후 completed 행수:", len(completed_df), ")")
print("matched_amount 채워진 행수:", instance["matched_amount"].notna().sum())
print("중복 transaction 플래그 합:", instance["dup_transaction_flag"].sum())

print("\n컬럼 목록:", list(instance.columns))
instance.to_csv("offer_instance_table.csv", index=False)
print("\n저장 완료: offer_instance_table.csv")

원본 shape: (10, 6) (17000, 5) (306534, 4)

[portfolio_전처리]
   reward  difficulty  duration     offer_type  \
0      10          10         7           bogo   
1      10          10         5           bogo   
2       0           0         4  informational   
3       5           5         7           bogo   
4       5          20        10       discount   

                           offer_id  is_web  is_email  is_mobile  is_social  \
0  ae264e3637204a6fb9bb56bc8210ddfd       0         1          1          1   
1  4d5c57ea9a6940dd891ad53e9dbe8da0       1         1          1          1   
2  3f207df678b143eea3cee63160fa8bed       1         1          1          0   
3  9b98b8c7a33c4b65b9aebfe6a799e6d9       1         1          1          0   
4  0b1e1539f2cc45b7b9fa7c272da2e1d7       1         1          0          0   

   duration_hours  reward_ratio  
0             168          1.00  
1             120          1.00  
2              96           NaN  
3             168          1.0